# 15a — Hopfield Network

Corresponds to: Note 15 — Hopfield Network

**Libraries introduced in this notebook:** No new libraries — pure Python only.

**What this notebook demonstrates:**
1. Constructing the weight matrix using the Hebbian outer product rule
2. Verifying the worked example from the note (N=4, two patterns)
3. Asynchronous update and retrieval from a noisy input
4. Spurious states — the reversed pattern as an attractor

## Part 1: Hebbian Weight Rule

The weight matrix is constructed as the sum of outer products of stored patterns, scaled by 1/N, with the diagonal zeroed.

We implement this with plain Python lists — no numpy.

In [ ]:
def outer_product(a, b):
    """Outer product of two lists: result[i][j] = a[i] * b[j]"""
    return [[a[i] * b[j] for j in range(len(b))] for i in range(len(a))]

def add_matrices(A, B):
    return [[A[i][j] + B[i][j] for j in range(len(A[0]))] for i in range(len(A))]

def scale_matrix(M, s):
    return [[M[i][j] * s for j in range(len(M[0]))] for i in range(len(M))]

def zero_diagonal(M):
    W = [row[:] for row in M]
    for i in range(len(W)):
        W[i][i] = 0.0
    return W

def build_weights(patterns):
    N = len(patterns[0])
    W = [[0.0] * N for _ in range(N)]
    for p in patterns:
        W = add_matrices(W, outer_product(p, p))
    W = scale_matrix(W, 1.0 / N)
    W = zero_diagonal(W)
    return W

def print_matrix(M, label):
    print(label)
    for row in M:
        print('  ' + '  '.join(f'{v:6.3f}' for v in row))
    print()

## Part 2: Worked Example from the Note

Store two patterns of length N = 4:

```
x1 = (+1, -1, +1, -1)
x2 = (+1, +1, -1, -1)
```

Expected weight matrix (from the note):

```
W = [  0     0     0   -0.5 ]
    [  0     0    -0.5  0   ]
    [  0    -0.5   0    0   ]
    [ -0.5   0     0    0   ]
```

In [ ]:
x1 = [+1, -1, +1, -1]
x2 = [+1, +1, -1, -1]
patterns = [x1, x2]

W = build_weights(patterns)
print_matrix(W, 'Weight matrix W (N=4, 2 patterns):')

## Part 3: Asynchronous Update and Retrieval

The update rule: for each neuron i, compute net_i = sum_j(W[i][j] * s[j]) and set s[i] = +1 if net_i >= 0, else -1.

We run one full pass through all neurons. If no state changes, the network has converged.

In [ ]:
def net_input(W, s, i):
    return sum(W[i][j] * s[j] for j in range(len(s)))

def update_neuron(W, s, i):
    net = net_input(W, s, i)
    return +1 if net >= 0 else -1

def run_async(W, s_init, max_steps=20):
    s = s_init[:]
    N = len(s)
    print(f'Initial state: {s}')
    for step in range(max_steps):
        changed = False
        for i in range(N):
            new_val = update_neuron(W, s, i)
            if new_val != s[i]:
                print(f'  Step {step+1}: neuron {i+1} flipped {s[i]:+d} -> {new_val:+d}  (net={net_input(W, s, i):.3f})')
                s[i] = new_val
                changed = True
        if not changed:
            print(f'Converged after {step+1} pass(es).')
            break
    print(f'Final state:   {s}')
    return s

# Noisy input: x1 with s1 flipped
s_noisy = [-1, -1, +1, -1]
print('--- Retrieval from noisy x1 (s1 flipped) ---')
s_final = run_async(W, s_noisy)
print(f'x1 = {x1}')
print(f'Match: {s_final == x1}')

## Part 4: Spurious States — The Reversed Pattern

The reversed pattern -x1 has the same energy as x1. Starting from -x1 should converge back to -x1 (it is already a fixed point).

In [ ]:
x1_reversed = [-v for v in x1]
print(f'x1          = {x1}')
print(f'-x1 (reversed) = {x1_reversed}')
print()
print('--- Starting from -x1 ---')
s_rev = run_async(W, x1_reversed)
print(f'Is -x1 a fixed point: {s_rev == x1_reversed}')

## Summary

This notebook demonstrated:
- The Hebbian outer product rule builds a weight matrix that encodes co-activation statistics of stored patterns
- The worked example from the note is reproduced exactly: W matches the hand-calculated matrix
- Asynchronous update recovers x1 from a one-bit-corrupted input in a single pass
- The reversed pattern -x1 is a fixed point — a spurious attractor with the same energy as x1